# Lekcija 13 - Memorija agenta


## Postavljanje

Ova bilježnica prikazuje kako izgraditi agenta za rezervaciju putovanja s **trajnom memorijom** koristeći **Microsoft Agent Framework** (MAF).

Naučit ćete kako različite vrste memorije agenta — radna, kratkoročna i dugoročna — utječu na to kako agent zadržava i koristi informacije tijekom razgovora.

**Preduvjeti:**
- Azure AI Foundry projekt s implementiranim chat modelom (npr. `gpt-4o-mini`).
- Prijava putem Azure CLI — pokrenite `az login` u svom terminalu.
- `AZURE_AI_PROJECT_ENDPOINT` — vaš krajnji Azure AI Foundry projekt.
- `AZURE_AI_MODEL_DEPLOYMENT_NAME` — naziv vašeg implementiranog modela.


In [ ]:
%pip install agent-framework azure-ai-projects azure-identity -q

In [ ]:
import logging
logging.getLogger("agent_framework.azure").setLevel(logging.ERROR)

import os
import json
from typing import Annotated
from datetime import datetime

from dotenv import load_dotenv

from agent_framework import tool
from agent_framework.azure import AzureAIProjectAgentProvider
from azure.identity import AzureCliCredential

load_dotenv()

In [ ]:
provider = AzureAIProjectAgentProvider(credential=AzureCliCredential())

print("✅ AzureAIProjectAgentProvider created")

## Vrste memorije agenta

AI agenti mogu koristiti različite vrste memorije, od kojih svaka služi različitoj svrsi:

### Radna memorija
Sama nit razgovora — poruke razmijenjene u jednoj sesiji. Agent se može pozvati na ranije poruke u istoj niti kako bi održao koherentnost. U MAF-u se ovo stvara putem **`agent.create_session()`**, što vraća `AgentSession`.

### Kratkoročna memorija
Informacije koje traju tijekom trajanja zadatka ili sesije, ali se ne pohranjuju trajno. Na primjer, agent može prikupiti činjenice tijekom višekratnog planiranja razgovora i koristiti ih za izradu konačnog itinerera.

### Dugoročna memorija
Preferencije i činjenice koje traju **među sesijama**. Povratni korisnik ne bi trebao ponavljati svoja prehrambena ograničenja ili stil putovanja. Dugoročna memorija obično se podržava vanjskom pohranom — bazom podataka, datotekom ili vektorskim indeksom — i prikazuje agentu putem alata.


## Radna memorija s sesijama

Najnjednostavniji oblik memorije je sesija razgovora. Kada isti sesiji (kreiranoj putem `agent.create_session()`) prosljeđujete u uzastopne pozive `agent.run()`, agent vidi cijelu povijest tog razgovora i može se prisjetiti ranijih detalja.

Kreirajmo putničkog agenta i demonstrirajmo radnu memoriju.


In [ ]:
agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelMemoryAgent",
    instructions=(
        "You are a travel agent who remembers user preferences across conversations. "
        "Track destinations mentioned, budget constraints, and travel dates."
    ),
)

session = agent.create_session()

# First message — the user shares preferences
response = await agent.run(
    "I love beach destinations and my budget is $3000",
    session=session,
)
print("Agent:", response)

# Second message — the agent should recall the budget from the thread
response = await agent.run(
    "What did I say my budget was?",
    session=session,
)
print("Agent:", response)

Agent se ispravno sjetio proračuna jer obje poruke dijele istu sesiju. Ovo je **radna memorija** — postoji samo tijekom trajanja sesije.

### Što se događa s novom temom?

Ako stvorimo **novu** sesiju, agent nema sjećanje na prethodni razgovor:


In [ ]:
new_session = agent.create_session()

response = await agent.run(
    "What is my budget?",
    session=new_session,
)
print("Agent:", response)
print("\n💡 The agent has no memory of the previous conversation — it's a fresh session.")

## Uzorak dugoročnog pamćenja

Za pamćenje korisničkih preferencija **kroz sesije**, potrebna nam je trajna pohrana koja postoji izvan niti razgovora. Agent pristupa toj pohrani putem **alata** — funkcija koje može pozvati da bi spremio i dohvatilo informacije.

Ispod implementiramo jednostavnu pohranu preferencija u memoriji (u produkciji biste to podržali bazom podataka ili vektorskim indeksom) i izlažemo je kao alate koje agent može koristiti.

### Arhitektura
```
┌─────────────────┐     ┌──────────────────┐     ┌─────────────────┐
│  MAF Agent      │────▶│  @tool functions  │────▶│  Preference     │
│  (LLM)          │     │  save / retrieve  │     │  Store (dict)   │
└─────────────────┘     └──────────────────┘     └─────────────────┘
         │                                                 │
    AgentSession                                   Persists across
    (working memory)                               sessions
```


In [ ]:
# --- Persistent preference store (simulated) ---
preference_store: dict[str, list[str]] = {}


@tool(approval_mode="never_require")
def save_preference(
    user_id: Annotated[str, "User identifier"],
    preference: Annotated[str, "A travel preference to remember"],
) -> str:
    """Save a user travel preference to long-term memory."""
    preference_store.setdefault(user_id, []).append(preference)
    return f"✅ Stored: {preference}"


@tool(approval_mode="never_require")
def get_preferences(
    user_id: Annotated[str, "User identifier"],
) -> str:
    """Retrieve all saved travel preferences for a user."""
    prefs = preference_store.get(user_id, [])
    if not prefs:
        return f"No saved preferences for {user_id}."
    return "Saved preferences:\n- " + "\n- ".join(prefs)


@tool(approval_mode="never_require")
def search_hotels(
    query: Annotated[str, "Search query — location, amenities, or tags"],
) -> str:
    """Search the hotel database for matching properties."""
    hotels = [
        {"name": "Le Meurice Paris", "location": "Paris, France", "price": 850, "tags": ["luxury", "romantic", "spa"]},
        {"name": "Four Seasons Maui", "location": "Maui, Hawaii", "price": 695, "tags": ["beach", "family", "resort"]},
        {"name": "Aman Tokyo", "location": "Tokyo, Japan", "price": 780, "tags": ["luxury", "city", "spa"]},
        {"name": "Hotel Sacher Vienna", "location": "Vienna, Austria", "price": 420, "tags": ["historic", "accessible", "cultural"]},
        {"name": "Fairmont Whistler", "location": "Whistler, Canada", "price": 380, "tags": ["ski", "family", "mountain"]},
    ]
    q = query.lower()
    matches = [
        h for h in hotels
        if q in h["name"].lower()
        or q in h["location"].lower()
        or any(q in t for t in h["tags"])
    ]
    if not matches:
        matches = hotels[:3]
    return json.dumps(matches, indent=2)


print("✅ Tools defined: save_preference, get_preferences, search_hotels")

### Scenario 1 — Korisnik koji prvi put dolazi rezervira putovanje za godišnjicu

Sarah posjećuje prvi put. Agent bi trebao spremiti njezine preferencije putem alata i koristiti ih za preporuku hotela.


In [ ]:
travel_agent = await provider.create_agent(
    tools=[save_preference, get_preferences],
    name="TravelBookingAssistant",
    instructions=(
        "You are a personalized travel booking assistant with long-term memory.\n"
        "WORKFLOW:\n"
        "1. When a user starts a conversation, call get_preferences() to check for saved information.\n"
        "2. Store any new preferences the user mentions using save_preference().\n"
        "3. Use search_hotels() to find suitable options that match their preferences and budget.\n"
        "4. Do NOT recommend hotels that exceed the user's budget.\n\n"
        "IMPORTANT: Always use user_id='sarah_johnson_123' for all memory operations."
    ),
)

session_1 = travel_agent.create_session()

response = await travel_agent.run(
    "Hi! I'm Sarah and I'm planning a trip for my 10th wedding anniversary. "
    "We love romantic destinations, fine dining, and spa experiences. "
    "My husband has mobility issues, so we need accessible accommodations. "
    "Our budget is around $700-800 per night.",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
response = await travel_agent.run(
    "The Hotel Sacher sounds perfect! We're both vegetarian and I have a "
    "severe nut allergy. Can you note that for future trips?",
    session=session_1,
)
print("🤖 Agent:", response)

In [ ]:
# Verify what was stored
print("📋 Preference store contents:")
for uid, prefs in preference_store.items():
    print(f"\n  User: {uid}")
    for p in prefs:
        print(f"    - {p}")

### Scenario 2 — Sarah se vraća tjednima kasnije

Sarah započinje **potpuno novu temu** (simulirajući novu sesiju). Radna memorija je prazna, ali spremište dugotrajnih preferencija još uvijek ima njezine podatke. Agent bi ih trebao dohvatiti i koristiti za personalizaciju preporuka.


In [ ]:
session_2 = travel_agent.create_session()  # New session — no working memory

response = await travel_agent.run(
    "Hi, my husband and I are planning another trip. Can you recommend a good hotel?",
    session=session_2,
)
print("🤖 Agent:", response)
print("\n💡 The agent retrieved Sarah's saved preferences from long-term memory "
      "even though this is a completely new conversation thread.")

In [ ]:
response = await travel_agent.run(
    "Great suggestions! For the Maui option, what activities would you recommend for the kids?",
    session=session_2,
)
print("🤖 Agent:", response)

## Sažetak

U ovoj lekciji naučili ste tri vrste memorije agenata i kako ih implementirati pomoću Microsoft Agent Frameworka:

| Vrsta memorije | MAF mehanizam | Trajanje |
|---|---|---|
| **Radna** | `agent.create_session()` | Jedan razgovor |
| **Kratkoročna** | Akumulirani kontekst unutar niti | Jedan zadatak / sesija |
| **Dugoročna** | Vanjska pohrana pristupljena putem `@tool` funkcija | Preko više sesija |

### Ključni zaključci
1. **`agent.create_session()`** osigurava radnu memoriju — agent vidi cjelokupnu povijest razgovora unutar sesije.
2. **Nove sesije gube kontekst** — bez dugoročne memorije agent ne može zapamtiti prošle razgovore.
3. **`@tool` funkcije** premošćuju jaz — omogućuju agentu pohranu i dohvat informacija iz trajne pohrane.
4. **Personalizacija se poboljšava tijekom vremena** — što se više preferencija pohranjuje, bolje su preporuke agenta.

### Primjene u stvarnom svijetu
- **Korisnička služba**: Pamćenje povijesti i preferencija korisnika
- **Osobni asistenti**: Održavanje konteksta tijekom dana ili tjedana
- **Zdravstvo**: Praćenje podataka i preferencija pacijenata
- **E-trgovina**: Personalizirana kupovina na temelju povijesti

### Sljedeći koraci
- Zamijeniti memoriju u rječniku s bazom podataka ili vektorskom pohranom (npr. Azure AI Search)
- Dodati isteka memorije za vremenski osjetljive informacije
- Izgraditi sustave s više agenata s dijeljenom memorijom
- Istražiti Cognee bilježnicu za memoriju podržanu znanstvenim grafom


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Odricanje od odgovornosti**:
Ovaj dokument je preveden pomoću AI prevoditeljskog servisa [Co-op Translator](https://github.com/Azure/co-op-translator). Iako se trudimo za točnost, imajte na umu da automatski prijevodi mogu sadržavati pogreške ili netočnosti. Izvorni dokument na izvornom jeziku treba smatrati autoritativnim izvorom. Za kritične informacije preporučuje se profesionalni ljudski prijevod. Ne odgovaramo za bilo kakve nesporazume ili kriva tumačenja nastala korištenjem ovog prijevoda.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
